<div style="background-color: #ffffff; color: #000000; padding: 10px;">
<div style="display: flex; justify-content: space-between; align-items: center; background-color: #ffffff; color: #000000; padding: 10px;">
    <img src="../00_aisc/img/logo_aisc_150dpi.png" height="80" style="margin-right: auto;" alt="Logo of the AI Service Center Berlin-Brandenburg.">
    <img src="../00_aisc/img/logo_bmftr_de.png" height="150" style="margin-left: auto;" alt="Logo of the German Federal Ministry of Research, Technology and Space: Gefoerdert vom Bundesministerium fuer Forschung, Technologie und Raumfahrt.">
</div>
<h1> Workshop: Rapid Prototyping via Vibe Coding
<h2> Part 1 &mdash; GEPA
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
    <h2> Part 1 &mdash; GEPA</h2>
</div>

**GEPA** (pronounced *ghee-pa*) is a Python library that **optimises any text parameter** &mdash; system prompts, code snippets, agent architectures, configuration strings &mdash; by combining LLM-based reflection with Pareto-efficient evolutionary search.

The paper claim: up to **35x fewer evaluations** than reinforcement-learning approaches for the same quality gain. The practical claim: when you have a metric function that scores a candidate, you no longer have to hand-tune prompts.

Repository: <https://github.com/gepa-ai/gepa> &middot; License: MIT.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. Why traditional prompt optimisers are weak</h3>
</div>

Classic optimisers know *that* a candidate failed (score = 0.3), not *why*. GEPA feeds **full execution traces** &mdash; error messages, intermediate outputs, profiling data &mdash; to a reflection LLM, which diagnoses failures and proposes *targeted* improvements to the next candidate.

This changes the shape of the search:

| Dimension | Random / grid search | GEPA |
|-----------|---------------------|------|
| Signal per failed run | `metric_score` only | `metric_score` + trace + reflection |
| Mutation strategy | random perturbation | LLM-guided edit ("the prompt missed edge case X; add...") |
| Selection | highest-scoring | Pareto-efficient across metric(s) |
| Typical eval budget | thousands | dozens to low hundreds |

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. Install</h3>
</div>

```bash
pip install gepa
# or, for the latest:
pip install git+https://github.com/gepa-ai/gepa.git
```

GEPA is already installed in this workshop's Python environment. Verify:

In [ ]:
import gepa

print("gepa imported successfully")
print("module path:", gepa.__file__)


<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. Minimal example</h3>
</div>

The simplest shape of a GEPA optimisation:

```python
import gepa

result = gepa.optimize(
    seed_candidate={"system_prompt": "You are a helpful assistant."},
    trainset=trainset,
    valset=valset,
    task_lm="openai/gpt-4.1-mini",
    max_metric_calls=150,
    reflection_lm="openai/gpt-5",
)
```

Three things define the run:

1. **`seed_candidate`** &mdash; the starting point. A dict whose keys name the text parameters being optimised.
2. **`trainset` / `valset`** &mdash; evaluation datasets. Each example is `(input, expected_output)` with an associated metric.
3. **`task_lm` / `reflection_lm`** &mdash; which model evaluates candidates (task) vs. which model critiques and proposes mutations (reflection). Reflection is usually worth a bigger model even if the task is small.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>4. Toy run you can execute (optional)</h3>
</div>

This cell is **not executed automatically** &mdash; it will make real LLM calls and cost a few cents. Skim it to understand the shape, and run if you want to see the numbers for yourself. You need an API key; `LITELLM_LOG=DEBUG` helps debug provider issues.

The example optimises a sentiment-classification prompt on a five-row toy set.

In [ ]:
# Set one of these before running (choose what you have a key for):
#   export ANTHROPIC_API_KEY=...
#   export OPENAI_API_KEY=...
#
# The `task_lm` / `reflection_lm` strings are LiteLLM provider/model identifiers.

import os

assert os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENAI_API_KEY"), (
    "set ANTHROPIC_API_KEY or OPENAI_API_KEY before running this cell"
)

import gepa

trainset = [
    {"input": "I loved this film",        "answer": "positive"},
    {"input": "what a waste of two hours","answer": "negative"},
    {"input": "it was fine, I guess",     "answer": "neutral"},
    {"input": "utterly brilliant",        "answer": "positive"},
    {"input": "boring from the first minute","answer": "negative"},
]
valset = [
    {"input": "better than I expected",   "answer": "positive"},
    {"input": "I wanted to walk out",     "answer": "negative"},
]

# See gepa.readthedocs.io for the full metric signature.
# This is a skeleton -- wire a real metric before running.
def metric(candidate, example):
    # ask task_lm with candidate["system_prompt"] and example["input"],
    # compare to example["answer"], return {"score": 0 or 1, "trace": "..."}
    raise NotImplementedError("fill in with a real call to the task_lm")

# result = gepa.optimize(
#     seed_candidate={"system_prompt": "Classify the sentiment as positive, negative, or neutral."},
#     trainset=trainset,
#     valset=valset,
#     task_lm="anthropic/claude-haiku-4-5",
#     reflection_lm="anthropic/claude-sonnet-4-6",
#     max_metric_calls=30,
#     metric=metric,
# )
# print(result.best_candidate)
print("scaffolded; uncomment and wire `metric` to run")


<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>5. Picking a reflection model</h3>
</div>

The reflection model does not run every iteration &mdash; it runs when GEPA wants a *diagnosis*. It's usually worth spending on a larger model here even if your task model is small:

| Setup | Best for |
|-------|----------|
| Task = small / fast model (Haiku, gpt-4.1-mini), Reflection = larger model (Sonnet, gpt-5) | Most production use. Cheap evals, smart mutations. |
| Task == Reflection == small | Early exploration, cheap, noisier |
| Task == Reflection == large | Maximum quality, high cost, rarely worth it |

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>6. DSPy integration (optional)</h3>
</div>

GEPA integrates with [DSPy](https://github.com/stanfordnlp/dspy) &mdash; a framework that treats LLM pipelines as modular, compilable programs. If you're already using DSPy, you can optimise any DSPy module's signatures and prompts with GEPA as the optimiser backend.

If you're *not* using DSPy, skip this. GEPA works fine as a standalone library.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>7. How to combine GEPA with Claude Code</h3>
</div>

Two patterns worth knowing:

1. **Optimise a skill or subagent's system prompt.**
   Copy the frontmatter + body of your best-performing skill into a GEPA seed candidate. Define a metric against a held-out task set. Run GEPA. Paste the winning prompt back into `.claude/skills/<name>/SKILL.md`.

2. **Optimise the *instructions* inside a scaffolded script.**
   E.g. the docling-toolkit scaffolder emits a Python script whose LLM-consumer prompt sits in a constant. GEPA-optimise *that* string against your evaluation set without touching Claude Code itself.

Rule of thumb: **GEPA earns its keep when you have a quantifiable metric and an inner loop that runs many times.** Use it for the hot path, not every prompt.